In [ ]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Multimodal"

In [ ]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


 # Image input

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
first_file_name = list(uploader.value.keys())[0]
uploaded_file = uploader.value[first_file_name]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
from langchain.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])



In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model=model
 )

In [ ]:
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

# Audio input

In [ ]:
!pip install sounddevice tqdm -q

!apt-get install libportaudio2 -q

In [ ]:
from ipywidgets import FileUpload

uploader = FileUpload(accept='.ogg', multiple=False)
display(uploader)

In [ ]:
import base64

# Check if any files have been uploaded via the uploader widget
if uploader.value:
    # Get the filename of the first uploaded audio file
    # (Since uploader.value is a dictionary where keys are filenames)
    audio_file_name = list(uploader.value.keys())[0]
    audio_file_data = uploader.value[audio_file_name]

    # Convert the 'memoryview' content to raw bytes, then encode to Base64 string
    # This format is required to send multimedia data to the Gemini API
    audio_bytes = bytes(audio_file_data["content"])
    aud_b64 = base64.b64encode(audio_bytes).decode("utf-8")

    print(f"✅ Audio file loaded successfully: {audio_file_name}")
else:
    # Handle the case where the user runs the cell without uploading a file
    print("❌ Error: Please upload an audio file (WAV or MP3) first.")

In [ ]:


multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

 [messages for multimodal](https://docs.langchain.com/oss/python/langchain/messages)